In [1]:
from sklearn.datasets import load_iris
import numpy as np

iris = load_iris()
X, y = iris.data, iris.target

In [3]:
X.shape

(150, 4)

In [2]:
print(np.unique(y))  # [0 1 2]

[0 1 2]


In [ ]:
# 3
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,    
    random_state=42   
)

print(X_train.shape) 
print(X_test.shape)

(120, 4)
(30, 4)


In [ ]:
# 4
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)   # faqat transform! (fit emas)

In [ ]:
# 5hci
import numpy as np
from collections import Counter

def euclidean(x1, x2):
    return np.sqrt(np.sum((x1 - x2) ** 2))

def knn_predict(X_train, y_train, x_new, k=3):
    distances = []
    
    for i in range(len(X_train)):
        dist = euclidean(X_train[i], x_new)
        distances.append((dist, y_train[i]))   
   
    distances.sort(key=lambda x: x[0])
    
    k_nearest = distances[:k]
    
    k_labels = [label for (dist, label) in k_nearest]
    
    vote = Counter(k_labels)
    return vote.most_common(1)[0][0]  

In [10]:
# 6
predictions = [knn_predict(X_train_scaled, y_train, x, k=3) 
               for x in X_test_scaled]

print( predictions[:10])
print(list(y_test[:10]))

[np.int64(1), np.int64(0), np.int64(2), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(2), np.int64(1), np.int64(1)]
[np.int64(1), np.int64(0), np.int64(2), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(2), np.int64(1), np.int64(1)]


In [12]:
# 7
def accuracy(y_true, y_pred):
    correct = sum(1 for true, pred in zip(y_true, y_pred) if true == pred)
    return correct / len(y_true)

acc = accuracy(y_test, predictions)
print(f"{acc:.4f} ({acc*100:.1f}%)")

1.0000 (100.0%)


In [ ]:
# 8chi
from sklearn.neighbors import KNeighborsClassifier

knn_sklearn = KNeighborsClassifier(n_neighbors=3)
knn_sklearn.fit(X_train_scaled, y_train)
sklearn_preds = knn_sklearn.predict(X_test_scaled)

sklearn_acc = accuracy(y_test, sklearn_preds)
print(f"{sklearn_acc:.4f} ({sklearn_acc*100:.1f}%)")
print(f"{acc:.4f} ({acc*100:.1f}%)")

1.0000 (100.0%)
1.0000 (100.0%)


In [ ]:
# 9chi
k_values = [1, 3, 5, 7, 9]
results = {}

for k in k_values:
    preds = [knn_predict(X_train_scaled, y_train, x, k=k) 
             for x in X_test_scaled]
    acc_k = accuracy(y_test, preds)
    results[k] = acc_k
    print(f"K={k}: {acc_k*100:.1f}%")

best_k = max(results, key=results.get)
print(f"K = {best_k} ({results[best_k]*100:.1f}%)")

K=1: 100.0%
K=3: 100.0%
K=5: 100.0%
K=7: 100.0%
K=9: 100.0%
K = 1 (100.0%)


In [ ]:
# 10hci
def knn_weighted(X_train, y_train, x_new, k=3):
    distances = []
    for i in range(len(X_train)):
        dist = euclidean(X_train[i], x_new)
        dist = max(dist, 1e-10)         
        distances.append((dist, y_train[i]))
    
    distances.sort(key=lambda x: x[0])
    k_nearest = distances[:k]
    
    weights = {}
    for dist, label in k_nearest:
        w = 1 / dist
        weights[label] = weights.get(label, 0) + w
    
    return max(weights, key=weights.get)

In [17]:
# 2ta feature bn
X_train_2f = X_train_scaled[:, [2, 3]]
X_test_2f  = X_test_scaled[:, [2, 3]]

preds_2f = [knn_predict(X_train_2f, y_train, x, k=3) 
            for x in X_test_2f]
acc_2f = accuracy(y_test, preds_2f)
print(f"2 feature accuracy: {acc_2f*100:.1f}%")
print(f"4 feature accuracy: {acc*100:.1f}%")

2 feature accuracy: 100.0%
4 feature accuracy: 100.0%
